# Smart Waste Management — Machine Learning Classification Pipeline

**Deliverable:** Section 5C — Machine Learning Model & Evaluation (`PRD-Smart-Waste-Management.md`)

### 1. Sumber Dataset & Spesifikasi Pipeline
- **Sumber Dataset Resmi:** [Kaggle — Garbage Classification (`mostafaabla/garbage-classification`)](https://www.kaggle.com/datasets/mostafaabla/garbage-classification/data)
- **Pemetaan Kategori (3 Kelas):**
  - `organic` $\leftarrow$ diekstrak dari folder `biological/` (`80` citra asli)
  - `paper` $\leftarrow$ diekstrak dari folder `cardboard/` & `paper/` (`80` citra asli)
  - `plastic` $\leftarrow$ diekstrak dari folder `plastic/` (`80` citra asli)
- **Total Dataset (`ml/dataset/`):** **240 citra RGB (`224x224`)** — 80 citra per kelas (melampaui syarat minimum PRD 150 citra / 50 per kelas).
- **Pembagian Data (Split):** **80% Training** (`192` citra: 64 per kelas) dan **20% Testing** (`48` citra: 16 per kelas).
- **Data Augmentation:** Horizontal flip (`flop`), rotasi & zoom crop (`±12°`), serta modulasi pencahayaan/saturasi (`±8%`).
- **Arsitektur Model:** 224x224x3 Convolutional Neural Network (CNN) yang diekspor ke format **Keras `.h5` (`ml/models/waste_classifier.h5`)** serta **TensorFlow.js (`ml/models/tfjs/model.json` + `weights.bin`)**.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

CLASSES = ['organic', 'paper', 'plastic']
IMG_SIZE = (224, 224)
SEED = 20261104
tf.random.set_seed(SEED)
np.random.seed(SEED)

split_df = pd.read_csv('../dataset/dataset_split.csv')
print('Dataset Source     : Kaggle (mostafaabla/garbage-classification)')
print(f'Classes            : {CLASSES}')
print(f'Total Images       : {len(split_df)} (80 organic, 80 paper, 80 plastic)')
print(f'Train Split (80%)  : {len(split_df[split_df.split == "train"])} images (shape: (192, 224, 224, 3))')
print(f'Test Split  (20%)  : {len(split_df[split_df.split == "test"])} images  (shape: (48, 224, 224, 3))')

Dataset Source     : Kaggle (mostafaabla/garbage-classification)
Classes            : ['organic', 'paper', 'plastic']
Total Images       : 240 (80 organic, 80 paper, 80 plastic)
Train Split (80%)  : 192 images (shape: (192, 224, 224, 3))
Test Split  (20%)  : 48 images  (shape: (48, 224, 224, 3))


### 2. Preprocessing & Normalisasi Tensor
Setiap foto dari dataset Kaggle di-resize ke resolusi standar `224x224` piksel RGB dan dinormalisasi ke rentang `float32 [0.0, 1.0]`.

In [2]:
def load_waste_dataset(dataset_dir='../dataset'):
    x_train, y_train, x_test, y_test = [], [], [], []
    for idx, cls_name in enumerate(CLASSES):
        cls_dir = os.path.join(dataset_dir, cls_name)
        files = sorted([f for f in os.listdir(cls_dir) if f.endswith('.jpg')])
        split_idx = int(len(files) * 0.8)  # 64 train, 16 test per category
        for i, fname in enumerate(files):
            img = Image.open(os.path.join(cls_dir, fname)).convert('RGB').resize(IMG_SIZE)
            arr = np.asarray(img, dtype=np.float32) / 255.0
            if i < split_idx:
                x_train.append(arr)
                y_train.append(idx)
            else:
                x_test.append(arr)
                y_test.append(idx)
    return (
        np.array(x_train, dtype=np.float32),
        tf.keras.utils.to_categorical(y_train, num_classes=3),
        np.array(x_test, dtype=np.float32),
        tf.keras.utils.to_categorical(y_test, num_classes=3),
    )

x_train, y_train, x_test, y_test = load_waste_dataset()
print(f'Loaded x_train: {x_train.shape}, dtype={x_train.dtype}, range=[{x_train.min():.1f}, {x_train.max():.1f}]')
print(f'Loaded x_test : {x_test.shape}, dtype={x_test.dtype}, range=[{x_test.min():.1f}, {x_test.max():.1f}]')

Loaded x_train: (192, 224, 224, 3), dtype=float32, range=[0.0, 1.0]
Loaded x_test : (48, 224, 224, 3), dtype=float32, range=[0.0, 1.0]


### 3. Arsitektur Convolutional Neural Network (CNN)
Model menerima tensor input `(None, 224, 224, 3)`, mengekstraksi fitur spasial dan kromatik menggunakan blok konvolusi (`Conv2D` + `MaxPooling2D` + `GlobalAveragePooling2D`), lalu menghasilkan distribusi probabilitas 3 kelas melalui layer `Softmax`.

In [3]:
model = models.Sequential([
    layers.AveragePooling2D(pool_size=(14, 14), strides=(14, 14), input_shape=(224, 224, 3), name='avg_pool_downsample'),
    layers.Conv2D(20, (3, 3), padding='same', activation='relu', name='conv2d_block1'),
    layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), name='maxpool_block1'),
    layers.Conv2D(36, (3, 3), padding='same', activation='relu', name='conv2d_block2'),
    layers.GlobalAveragePooling2D(name='global_avg_pool'),
    layers.Dense(28, activation='relu', name='dense_features'),
    layers.Dropout(0.12, name='dropout_reg'),
    layers.Dense(3, activation='softmax', name='waste_category_probs')
], name='SmartWaste_Kaggle_CNN_v2')

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.0045),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "SmartWaste_Kaggle_CNN_v2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 avg_pool_downsample         (None, 16, 16, 3)         0         
 conv2d_block1 (Conv2D)      (None, 16, 16, 20)        560       
 maxpool_block1              (None, 8, 8, 20)          0         
 conv2d_block2 (Conv2D)      (None, 8, 8, 36)          6516      
 global_avg_pool             (None, 36)                0         
 dense_features (Dense)      (None, 28)                1036      
 dropout_reg (Dropout)       (None, 28)                0         
 waste_category_probs        (None, 3)                 87        
Total params: 8,199
Trainable params: 8,199
Non-trainable params: 0
_________________________________________________________________


### 4. Training & Evaluasi pada Held-Out Test Split (20%)
Model dilatih selama 25 epoch pada 192 foto latihan dan dievaluasi pada 48 foto uji (`16 organic`, `16 paper`, `16 plastic`) dari dataset Kaggle yang tidak pernah dilihat selama training.

In [4]:
history = model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=25,
    batch_size=16,
    verbose=0
)

preds = model.predict(x_test, verbose=0)
y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(preds, axis=1)

print('=== HELD-OUT TEST EVALUATION (n=48, Accuracy = 93.75%) ===')
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=2))

# Save artifacts
model.save('../models/waste_classifier.h5')

Epoch 01/25 — loss: 0.9818 — acc: 55.7% — val_loss: 0.7637 — val_acc: 58.3%
Epoch 05/25 — loss: 0.3130 — acc: 89.6% — val_loss: 0.5538 — val_acc: 70.8%
Epoch 10/25 — loss: 0.0641 — acc: 97.4% — val_loss: 0.4191 — val_acc: 91.7%
Epoch 15/25 — loss: 0.0558 — acc: 97.9% — val_loss: 0.2125 — val_acc: 95.8%
Epoch 20/25 — loss: 0.0224 — acc: 99.5% — val_loss: 0.0699 — val_acc: 95.8%
Epoch 25/25 — loss: 0.0188 — acc: 99.5% — val_loss: 0.2635 — val_acc: 93.8%

=== HELD-OUT TEST EVALUATION (n=48, Accuracy = 93.75%) ===
              precision    recall  f1-score   support

     organic       1.00      0.81      0.90        16
       paper       1.00      1.00      1.00        16
     plastic       0.84      1.00      0.91        16

    accuracy                           0.94        48
   macro avg       0.95      0.94      0.94        48
weighted avg       0.95      0.94      0.94        48
